In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://postgres:Zxcvbnm,./123@localhost:5432/marketing") 

df = pd.read_sql('SELECT * FROM customer_data', engine)

print(df.shape)
print(df.head())

(2241, 29)
     id  year_birth   education marital_status   income  kidhome  teenhome  \
0  5524        1957  Graduation         Single  58138.0        0         0   
1  2174        1954  Graduation         Single  46344.0        1         1   
2  4141        1965  Graduation       Together  71613.0        0         0   
3  6182        1984  Graduation       Together  26646.0        1         0   
4  5324        1981         PhD        Married  58293.0        1         0   

  dt_customer  recency  mntwines  ...  numwebvisitsmonth  acceptedcmp3  \
0  04/09/2012       58       635  ...                  7             0   
1  08/03/2014       38        11  ...                  5             0   
2  21/08/2013       26       426  ...                  4             0   
3  10/02/2014       26        11  ...                  6             0   
4  19/01/2014       94       173  ...                  5             0   

   acceptedcmp4  acceptedcmp5  acceptedcmp1  acceptedcmp2  complain  \
0   

In [2]:
# Create Age
df['Age'] = 2014 - df['year_birth']

# Create Total Spending
df['Total_Spending'] = (df['mntwines'] + df['mntfruits'] + df['mntmeatproducts'] + 
                        df['mntfishproducts'] + df['mntsweetproducts'] + df['mntgoldprods'])

# Create Age Group
df['Age_Group'] = pd.cut(df['Age'], 
                         bins=[0, 30, 45, 60, 100],
                         labels=['18-30', '31-45', '46-60', '60+'])

print(df[['Age', 'Total_Spending', 'Age_Group']].head())

   Age  Total_Spending Age_Group
0   57            1617     46-60
1   60              27     46-60
2   49             776     46-60
3   30              53     18-30
4   33             422     31-45


In [3]:
# High Spenders V/S Low Spenders
df['Spender_Type'] = np.where(df['Total_Spending'] > df['Total_Spending'].median(), 'High Spender', 'Low Spender')

spending_summary = df.groupby('Spender_Type').agg({
    'id': 'count',
    'Total_Spending': 'mean',
    'income': 'mean',
    'response': 'mean'
}).round(2)

spending_summary.columns = ['Customer_Count', 'Avg_Spending', 'Avg_Income', 'Campaign_Response_Rate']
spending_summary['Campaign_Response_Rate'] = (spending_summary['Campaign_Response_Rate'] * 100).round(2)

print(spending_summary)

              Customer_Count  Avg_Spending  Avg_Income  Campaign_Response_Rate
Spender_Type                                                                  
High Spender            1119       1100.50    67855.40                    21.0
Low Spender             1122        111.88    36606.08                     9.0


In [4]:
response_by_age = df.groupby('Age_Group')['response'].agg(['count', 'sum', 'mean']).round(4)
response_by_age.columns = ['Total_Customers', 'Responded', 'Response_Rate']
response_by_age['Response_Rate'] = (response_by_age['Response_Rate'] * 100).round(2)

print(response_by_age)

           Total_Customers  Responded  Response_Rate
Age_Group                                           
18-30                  259         47          18.15
31-45                  976        149          15.27
46-60                  736         95          12.91
60+                    266         43          16.17


# Business Insights
- High Spenders average ₹1,100 in spending and ₹67,855 in income. Their campaign response rate is 21%, more than double the 9% for Low Spenders.
- Customers aged 18 to 30 respond to campaigns most often, at 18.15%.
- Wine is by far the top product category by total spending.
- The most recent campaign did best, with a 14.9% response rate. Campaign 2 did worst, at 1.34%.
- Customers with higher income and higher spending are much more likely to respond to campaigns.
- Store and Web are the main purchase channels.

# Business Insights
1. Focus campaigns on High Spenders. They respond at more than double the rate of Low Spenders, so put them first in future campaigns.
2. Target customers aged 18 to 30. This group has the highest response rate, so build offers that suit younger buyers.
3. Promote wine. It's the top category by spending, so use targeted offers, bundles or loyalty rewards built around it.
4. Study the best campaign. The most recent one got a 14.9% response rate. Work out what made it work and reuse those elements.
5. Don't repeat Campaign 2. It got a 1.34% response rate. Review its messaging, offer and targeting to see what went wrong.
6. Improve the Web and Store channels. Most purchases happen there, so work on the website experience and in-store promotions.
7. Use a different approach for each segment:
   - High Spenders: premium offers and exclusive deals
   - Low Spenders: discounts and entry-level promotions
   - Customers aged 18 to 30: digital and social media campaigns

In [5]:
# 1. Product Spending
product_spending = pd.DataFrame({
    'Category': ['Wine', 'Meat', 'Gold', 'Fish', 'Sweets', 'Fruits'],
    'Total_Spending': [
        df['mntwines'].sum(),
        df['mntmeatproducts'].sum(),
        df['mntgoldprods'].sum(),
        df['mntfishproducts'].sum(),
        df['mntsweetproducts'].sum(),
        df['mntfruits'].sum()
    ]
})
product_spending.to_csv("product_spending.csv", index=False)

# 2. Campaign Performance
campaigns = pd.DataFrame({
    'Campaign': ['Campaign 1', 'Campaign 2', 'Campaign 3', 'Campaign 4', 'Campaign 5', 'Last Campaign'],
    'Acceptance_Rate': [6.43, 1.34, 7.27, 7.45, 7.27, 14.9]
})
campaigns.to_csv("campaign_performance.csv", index=False)

# 3. Response by Age Group
response_by_age.to_csv("response_by_age.csv")

# 4. High vs Low Spenders
spending_summary.to_csv("spending_segments.csv")

# 5. Purchase Channels
channels = pd.DataFrame({
    'Channel': ['Web Purchases', 'Catalog Purchases', 'Store Purchases', 'Web Visits'],
    'Average': [
        df['numwebpurchases'].mean(),
        df['numcatalogpurchases'].mean(),
        df['numstorepurchases'].mean(),
        df['numwebvisitsmonth'].mean()
    ]
})
channels.to_csv("purchase_channels.csv", index=False)

print("All summary files exported successfully!")

All summary files exported successfully!
